In [1]:
import random
%load_ext autoreload
%autoreload 2

import pandas as pd
pd.options.display.max_columns = 100

import pickle
import numpy as np
import matplotlib.pyplot as plt
from tqdm import tqdm
from datetime import datetime
import seaborn as sns
from collections import Counter
import json

import all_blocks
from merge_contracts import ContractsMerger
from generate_test import generate_preds
from get_distribution_utils import get_disrib_sums, cost_columns_to_datetime


c:\users\никита\appdata\local\programs\python\python39\lib\site-packages\pandas\core\computation\expressions.py:21: UserWarning: Pandas requires version '2.8.4' or newer of 'numexpr' (version '2.8.1' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED


## Подгружаем и обрабатываем датку

In [2]:
pays_df1 = pd.read_excel("data/Счета на оплату 3800-2023.XLSX")
pays_df2 = pd.read_excel("data/Счета на оплату 4200-4000-3800-2024.XLSX")
pays_df3 = pd.read_excel("data/Счета на оплату 5400-2023.XLSX")
pays_df4 = pd.read_excel("data/Счета на оплату 5400-2024.XLSX")
pays_df5 = pd.read_excel("data/Счета на оплату 5500-2023.XLSX")

In [3]:
pays_df = pd.concat([pays_df1, pays_df2, pays_df3, pays_df4, pays_df5], axis=0)

In [4]:
merger_df = pd.read_excel("data/Связь договор - здания.XLSX")
main_costs_df = pd.read_excel("data/Основные средства.XLSX")
squares = pd.read_excel("data/Площади зданий.XLSX")
serv_codes = pd.read_excel("data/Коды услуг.XLSX")

In [5]:
main_costs_df["тест_фича_нум"] = [random.randint(0, 100) for i in range(len(main_costs_df))]
main_costs_df["тест_фича_бул"] = [random.randint(0, 1) for i in range(len(main_costs_df))]
main_costs_df["тест_фича_дата"] = ["19.04.2006"] * len(main_costs_df)

In [6]:
merger = ContractsMerger(pays_df, merger_df, main_costs_df, squares, serv_codes)

E:\DIR_python_projects\ledaer_digital_transformation_24\zakupai\services\api\ml\lib\merge_contracts.py:78: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  self.needed_pays_df["time"] = self.needed_pays_df["Год"].map(lambda x: datetime(year=x, month=1, day=1))


In [7]:
import time

t = time.time()
res, features = merger.start_merging()
print(time.time() - t)

81.13339138031006


In [8]:
len(res)

105904

In [10]:
# сохраняем результат с 1 страницы загрузки данных
res.to_csv("res_datetimes.csv")
with open("features.pkl", "wb") as f:
    pickle.dump(features, f)

res остается лежать на бэке где-то
features нужны на фронте чтобы блоки с признаками создать и на бэке дальше
то есть там всегда будут 3 блока условия + блоки действий + feature из этого файла

## Подгружаем граф, запускаем распределение

In [11]:
features = pickle.load(open("features.pkl", "rb"))
res = pd.read_csv("res_datetimes.csv", index_col=0)

C:\Users\843E~1\AppData\Local\Temp/ipykernel_18224/930177580.py:2: DtypeWarning: Columns (22) have mixed types. Specify dtype option on import or set low_memory=False.
  res = pd.read_csv("res_datetimes.csv", index_col=0)


In [12]:
numeric_features = [i[0] for i in features if i[2] != "date"]
date_features = [i[0] for i in features if i[2] == "date"]

res = cost_columns_to_datetime(res, date_features)

In [69]:
res_by_prime = res.groupby("prime")
raw_graph = json.load(open("graphs/graph (6).json", "r"))

# TMP
# raw_graph["nodes"][2]["type"] = "use_feature_distribution"

In [101]:
all_blocks.init_graph(raw_graph, features)

In [121]:
unique_primes = res["prime"].unique()
distrib_sums = get_disrib_sums(res_by_prime, res["prime"].unique(), numeric_features)

100%|██████████| 4793/4793 [00:10<00:00, 449.76it/s]


In [122]:
preds = generate_preds(pays_df, serv_codes, res, distrib_sums)

100%|██████████| 4793/4793 [00:09<00:00, 507.11it/s]


In [244]:
preds.to_csv("Результат_распределенные_счета.csv")

In [125]:
# preds["Номер счета"] = preds["Номер счета"].astype(str)
# preds[preds["Номер счета"] == "5006241237"]

,Компания,Год счета,Номер счета,Позиция счета,Номер позиции распределения,Дата отражения в учетной системе,ID договора,Услуга,Класс услуги,Здание,Класс ОС,ID основного средства,"Признак ""Использование в основной деятельности""","Признак ""Способ использования""",Площадь ОС,Сумма распределения,Счет главной книги,ID услуги
89075,5500,2023,5006241237,1,0,2023-03-09 00:00:00,ДПН 5500/165440,800003262,S004,ЗДН 5500/1/3905,60804001,55006080400468400,1,0,3848.6,117017.86,7048209010,S004
89076,5500,2023,5006241237,1,1,2023-03-09 00:00:00,ДПН 5500/165440,800003262,S004,ЗДН 5500/1/3905,60804001,55006080400468401,0,0,0.0,0.00,7048209010,S004
89077,5500,2023,5006241237,1,2,2023-03-09 00:00:00,ДПН 5500/165440,800003262,S004,ЗДН 5500/1/3914,60401018,55006040054136050,1,0,88.6,62.02,7048209010,S004
89078,5500,2023,5006241237,1,3,2023-03-09 00:00:00,ДПН 5500/165440,800003262,S004,ЗДН 5500/1/3915,60401018,55006040054136060,1,0,243.9,469.97,7048209010,S004


In [127]:
preds["Счет главной книги"].unique()

array([7048208010, 7048414960, 7048406010, 7048209010, 7047505010,
       7048209020, 7047505020, 7048414970, 7047504010, 7048208020,
       7047504020, 7048406020, 7048401020, 7047805010, 7048401010],
      dtype=int64)